In [1]:
# Parameters
BATCH_MODE = "true"

# Complexity-03e — Barrières d'exposants et word RAM : lire une formalisation à la machine

**Série Complexity — accrétion de 03** · Public : **Recherche** · Hommage : **Josh Alman & Virginia Vassilevska Williams** (*Truly Subquadratic 3SUM and Truly Subcubic APSP*, octobre 2026), et le projet [anthropics/formal-math · 3sum-apsp](https://github.com/anthropics/formal-math/tree/main/3sum-apsp) qui les formalise.

**Navigation** : [Index de la série](README.md) | [03 — Plus de temps, plus de problèmes](Complexity-03-TimeHierarchy.ipynb) | [03b — Hartmanis & Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) | [03c — Zoo et oracles](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) | [07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb)

> **Bandeau d'honnêteté.** Ce carnet lit une formalisation **publiée par les auteurs** et n'exécute **aucun** de leurs programmes. Les cinq claims sont des **existences de programme** sur un word RAM défini dans le fichier même qui les énonce ; leurs constantes de temps ne sont pas optimisées (« large, and for most theorems no value is given » — README du projet). La mesure de ce carnet (le contrôle quadratique) est un **témoin du point de départ** : elle ne confirme ni ne réfute la borne 1,9992. Aucun `lake build` n'est lancé ici.

## 0. Pourquoi ce carnet

Le [07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) a raconté le contrepoint Williams–Alman du lundi 6 octobre : 3SUM en O(n^1.9992), APSP en O(n^2.9995), SETH intacte. Le [03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) a lu la réfutation annoncée de l'ETH par les mêmes vagues d'octobre. Ce carnet prend l'autre bout de la lorgnette : pas le récit, pas l'énoncé — **le modèle de calcul**.

La formalisation [`3sum-apsp`](https://github.com/anthropics/formal-math/tree/main/3sum-apsp) énonce ses cinq claims dans **un seul fichier de 139 lignes qui n'importe rien** — `EndStatement.lean` — et le README du projet pose la règle du jeu : « For the five claims this one file is all that has to be read. » Ce fichier définit sa propre machine (un word RAM de neuf instructions), sa propre sémantique de « résolu en temps O(n^r) », et cinq propositions. C'est une réponse **externe** au trou que la série documente depuis 03b : Mathlib n'a pas de couche `Computability/Complexity`, pas de `TIME(f)` — ici, le trou est comblé hors Mathlib, dans le noyau Lean seul.

Ce carnet fait trois choses : (1) reproduire les cinq claims avec leurs exposants **tels qu'imprimés** ; (2) **réimplémenter la machine en Python** et la vérifier sur un témoin fermé — une instance dont le compte de pas est calculé à la main, cellule par cellule ; (3) mesurer le point de départ — le 3SUM quadratique — en disant explicitement pourquoi la borne 1,9992 ne se mesure pas aujourd'hui.

In [2]:
import math
import random
import time
from fractions import Fraction

SEED = 20261028  # tout le carnet est rejouable a l'identique
print(f"Imports OK : math, random, time, fractions. SEED = {SEED}.")

Imports OK : math, random, time, fractions. SEED = 20261028.


## 1. Les cinq claims, tels qu'énoncés

Source : [`EndStatement.lean`](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/EndStatement.lean) (lignes 76–137) et la table du [README](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/README.md) du projet. Chaque claim est une **proposition Lean** (`def ... : Prop`) ; chacune est prouvée par un théorème du projet (`endStatement_*`) — la preuve vit hors de ce fichier, et ce carnet ne la recharge pas : il lit l'énoncé.

In [3]:
CLAIMS = [
    # (nom Lean, item papier, probleme, borne telle qu'imprimee, theoreme qui dit que le claim tient)
    ("Theorem_19", "Theorem 19", "Exact Triangle", "O(n^(3-0.0017))", "endStatement_theorem_19"),
    ("Theorem_22_3SUM", "Theorem 22", "3SUM", "O(n^1.9992)", "endStatement_theorem_22_3SUM"),
    ("Theorem_22_MinPlus", "Theorem 22", "produit (min,+)", "O(n^2.99942)", "endStatement_theorem_22_MinPlus"),
    ("Theorem_22_APSP", "Theorem 22", "APSP", "O(n^2.99942)", "endStatement_theorem_22_APSP"),
    ("Corollary_39_ZeroWeight", "Corollaire 39", "k-Clique de poids nul", "O(n^(k-0.0017*floor(k/3)))", "endStatement_corollary_39_zeroWeight"),
]
print(f"{'Claim Lean':24s} {'Papier':14s} {'Probleme':22s} {'Borne':28s} Theoreme")
print("-" * 108)
for c in CLAIMS:
    print(f"{c[0]:24s} {c[1]:14s} {c[2]:22s} {c[3]:28s} {c[4]}")
print()
# EndStatement.lean, lignes 48-51 : le noyau Lean n'a pas de puissances fractionnaires,
# donc BigO releve les deux membres a la puissance r.den.
for r in ["1.9992", "2.99942"]:
    f = Fraction(r)
    print(f"r = {r} = {f.numerator}/{f.denominator} :  BigO dit  T(n)^{f.denominator} <= K * n^{f.numerator}  pour n >= 2")

Claim Lean               Papier         Probleme               Borne                        Theoreme
------------------------------------------------------------------------------------------------------------
Theorem_19               Theorem 19     Exact Triangle         O(n^(3-0.0017))              endStatement_theorem_19
Theorem_22_3SUM          Theorem 22     3SUM                   O(n^1.9992)                  endStatement_theorem_22_3SUM
Theorem_22_MinPlus       Theorem 22     produit (min,+)        O(n^2.99942)                 endStatement_theorem_22_MinPlus
Theorem_22_APSP          Theorem 22     APSP                   O(n^2.99942)                 endStatement_theorem_22_APSP
Corollary_39_ZeroWeight  Corollaire 39  k-Clique de poids nul  O(n^(k-0.0017*floor(k/3)))   endStatement_corollary_39_zeroWeight

r = 1.9992 = 2499/1250 :  BigO dit  T(n)^1250 <= K * n^2499  pour n >= 2
r = 2.99942 = 149971/50000 :  BigO dit  T(n)^50000 <= K * n^149971  pour n >= 2


### Lecture du résultat — deux numéros par théorème, le claim prend le plus petit

Deux détails que la table seule ne montre pas, lus à la source :

- **Theorem 22 imprime deux bornes par problème** (papier, p. 36) : celle de l'algorithme passant par Theorem 5 — n^2−1/1296+o(1) ≤ O(n^1.99923) pour 3SUM, Õ(n^3−1/1944) ≤ O(n^2.99949) pour (min,+) et APSP — et celle de l'algorithme passant par Corollary 26. Le README du projet le dit : « The claims on these three problems state the numerals of the second algorithm, which are smaller » — les claims formels portent **1,9992 et 2,99942**, les bornes les plus fortes. Le résumé du papier (p. 1) arrondit l'autre algorithme : O(n^2.9995).
- **Le BigO du fichier est rationnel** : le noyau Lean n'a pas de puissances fractionnaires, alors `BigO` relève les deux membres à la puissance `r.den` — pour r = 1,9992 = 2499/1250, la borne littérale est « T(n)^1250 ≤ K·n^2499 pour n ≥ 2 ». Aucun exposant n'est arrondi au passage : ce que Lean prouve est exactement ce que la table affiche.

Et Exact Triangle : 3 − 0,0017 = 2,9983 — le papier lui-même (p. 63) décrit la formalisation comme « Theorem 19 (Exact Triangle in O(n^2.9983) time) ».

## 2. La machine : neuf instructions, un pas chacune

`EndStatement.lean` (lignes 15–46) définit la machine **dans l'énoncé même** :

- **Neuf instructions**, chacune coûte **un pas** (l'arrêt inclus) : écrire la constante 1 dans une cellule (`one`) ; additionner, soustraire, multiplier deux cellules modulo 2^W (`add`, `sub`, `mul`) ; charger et stocker **à adresse indirecte** signée (`load`, `store`) ; sauter à une position **fixée dans le texte du programme** si une cellule est négative (`bltz`) ; `accept` ; `reject`.
- **Mémoire** : cellules nommées par des entiers — négatifs compris — tenant des mots de W bits ; hors entrée, tout vaut 0 ; l'entrée se place en cellules 0, 1, 2, … et la **taille n en cellule 0 devant** (`loadWords`, `Problem.SolvedBy`).
- **Sortie du programme** : une position hors programme compte comme `reject` (`P.getD pc .reject`, ligne 33).
- **Ce que la machine n'a pas** (README : « There is no division, no shift, no bitwise operation, and no constant other than 1 ») : ni division, ni décalage, ni opération bit à bit, et **la seule constante est 1** — le zéro lui-même se fabrique par `sub x x`, un entier k par une chaîne de doubles en ~log k pas.

La cellule suivante porte ce modèle en Python — un **port de lecture**, fidèle à la lettre de `exec` (lignes 29–42), pas une preuve : ce qui rassure ici, c'est Lean, pas Python.

In [4]:
def mot_non_signe(v, W):
    return v & ((1 << W) - 1)

def signe(v, W):
    """Lecture en complement a deux -- docs/MACHINE.md : « a word is read as a
    signed number in two's complement wherever a sign matters »."""
    v &= (1 << W) - 1
    return v - (1 << W) if v >= (1 << (W - 1)) else v

def exec_wordram(P, mem0, W, t_max):
    """Port Python de exec (EndStatement.lean, lignes 29-42).
    P : liste d'instructions ; mem0 : dict adresse signee -> mot.
    Renvoie (verdict, memoire, pas) ou None si pas de halte en t_max pas.
    Chaque instruction coute un pas, l'arret inclus ; une position hors
    programme compte comme reject (P.getD pc .reject)."""
    mem = dict(mem0)
    pc, pas = 0, 0
    while pas < t_max:
        pas += 1
        instr = P[pc] if 0 <= pc < len(P) else ("reject",)
        op = instr[0]
        if op == "one":
            mem[instr[1]] = 1
            pc += 1
        elif op == "add":
            mem[instr[1]] = mot_non_signe(mem.get(instr[2], 0) + mem.get(instr[3], 0), W)
            pc += 1
        elif op == "sub":
            mem[instr[1]] = mot_non_signe(mem.get(instr[2], 0) - mem.get(instr[3], 0), W)
            pc += 1
        elif op == "mul":
            mem[instr[1]] = mot_non_signe(mem.get(instr[2], 0) * mem.get(instr[3], 0), W)
            pc += 1
        elif op == "load":
            mem[instr[1]] = mem.get(signe(mem.get(instr[2], 0), W), 0)
            pc += 1
        elif op == "store":
            mem[signe(mem.get(instr[1], 0), W)] = mem.get(instr[2], 0)
            pc += 1
        elif op == "bltz":
            pc = instr[2] if signe(mem.get(instr[1], 0), W) < 0 else pc + 1
        elif op == "accept":
            return (True, mem, pas)
        else:  # reject
            return (False, mem, pas)
    return None

def charger_mots(W, ws):
    """loadWords (ligne 45) : les mots d'entree en cellules 0, 1, 2, ... ;
    toute autre cellule (adresses negatives comprises) vaut 0."""
    m = {}
    for a, v in enumerate(ws):
        m[a] = mot_non_signe(v, W)
    return m

print("Interpreteur word RAM charge : 9 instructions, 1 pas chacune, arret inclus.")

Interpreteur word RAM charge : 9 instructions, 1 pas chacune, arret inclus.


### Lecture du résultat — le port, ce qu'il garantit et ce qu'il ne garantit pas

Le port suit `exec` instruction par instruction : un pas par instruction, arrêt inclus ; l'adresse hors programme devient `reject` ; `load`/`store` lisent l'adresse comme la valeur **signée** du mot ; les arithmétiques **enroulent modulo 2^W** (complément à deux). Ce que le port ajoute et que Lean n'a pas besoin d'écrire : un budget `t_max` de pas — le `t` de `exec`, décrémenté par la récursion chez Lean, bouclé chez nous.

Ce port ne prouve rien : il sert à **exécuter** des programmes de la machine pour les voir tourner. La preuve que les cinq claims tiennent vit dans `ThreeSumApsp/` du projet, chargée par `lake build` — que ce carnet ne lance pas.

## 3. Le témoin fermé : un décideur 3SUM pour n = 3, compté à la main

Pour vérifier la machine, le carnet écrit le plus petit décideur honnête : **3SUM pour exactement trois nombres** (mise en page de `SolvedBy` : n = 3 en cellule 0, les nombres en cellules 1, 2, 3). La machine n'ayant **ni test d'égalité ni constante 0**, tester s = a+b+c = 0 se fait en deux sauts : rejeter si s < 0, rejeter si −s < 0, sinon s = 0.

| Pos. | Instruction | Effet | Pas cumulés |
|---|---|---|---|
| 0 | `add 10, 1, 2` | [10] := a + b | 1 |
| 1 | `add 10, 10, 3` | [10] := s = a+b+c | 2 |
| 2 | `sub 11, 10, 10` | [11] := 0 (la seule façon d'obtenir 0) | 3 |
| 3 | `sub 12, 11, 10` | [12] := −s | 4 |
| 4 | `bltz 10, 7` | si s < 0 → position 7 | 5 |
| 5 | `bltz 12, 7` | si −s < 0 (s > 0) → position 7 | 6 |
| 6 | `accept` | s = 0 : accepter | 7 |
| 7 | `reject` | | |

**Compte à la main, chemin par chemin.** Instance **oui** (s = 0) : les deux `bltz` ne sautent pas, `accept` est le 7ᵉ pas — **7 pas**. Instance **non, s < 0** : le premier `bltz` (5ᵉ pas) saute en 7, `reject` est le 6ᵉ pas — **6 pas**. Instance **non, s > 0** : le second `bltz` (6ᵉ pas) saute en 7, `reject` est le 7ᵉ pas — **7 pas**.

La cellule suivante demande à l'interpréteur exactement ces trois comptes — et rien d'autre ne compte : si la machine et la main divergent d'un pas, le témoin échoue.

In [5]:
# Le decideur 3SUM pour n = 3, tel que compte a la main ci-dessus.
P_3SUM = [
    ("add",  10, 1, 2),    # 0 : [10] := a + b
    ("add",  10, 10, 3),   # 1 : [10] := s = a + b + c
    ("sub",  11, 10, 10),  # 2 : [11] := 0
    ("sub",  12, 11, 10),  # 3 : [12] := 0 - s
    ("bltz", 10, 7),       # 4 : si s < 0  -> reject
    ("bltz", 12, 7),       # 5 : si -s < 0 -> reject
    ("accept",),           # 6 : s = 0
    ("reject",),           # 7
]

def temoin(nombres, verdict_attendu, pas_attendu, W=16):
    m0 = charger_mots(W, [3] + list(nombres))
    res = exec_wordram(P_3SUM, m0, W, 100)
    verdict, _, pas = res
    ok = (verdict == verdict_attendu) and (pas == pas_attendu)
    print(f"  {str(nombres):16s} -> verdict={verdict!s:5s} pas={pas}"
          f" (attendu : verdict={verdict_attendu!s:5s}, pas={pas_attendu})"
          f"  [{'OK' if ok else 'ECHEC'}]")
    return ok

print("Trois cas canoniques, un par chemin du programme :")
cas = [
    ([-5, 2, 3],   True,  7),   # s = 0 : chemin accept
    ([-3, -2, -1], False, 6),   # s < 0 : saut au 5e pas
    ([ 1,  2,  3], False, 7),   # s > 0 : saut au 6e pas
]
tous_ok = all(temoin(nb, vd, pp) for nb, vd, pp in cas)

# Batterie seedee : le verdict de la machine contre la definition ThreeSum de la
# source (ligne 92 : trois positions DEUX A DEUX DISTINCTES ; a n = 3, l'unique triple).
random.seed(SEED)
accords, essais = 0, 300
for _ in range(essais):
    nb = [random.randint(-20, 20) for _ in range(3)]
    if random.random() < 0.35:
        i, j = random.sample(range(3), 2)
        nb[3 - i - j] = -(nb[i] + nb[j])   # forcer des cas oui
    attendu = (nb[0] + nb[1] + nb[2] == 0)
    m0 = charger_mots(16, [3] + nb)
    verdict, _, _ = exec_wordram(P_3SUM, m0, 16, 100)
    accords += (verdict == attendu)
print(f"Batterie seedee : {accords}/{essais} accords avec la definition de la source.")
print(f"TEMOIN FERME : {'CONFIRME' if tous_ok and accords == essais else 'ECHEC'}"
      " -- le compte de pas de l'interpreteur egale le compte a la main sur les trois chemins.")

Trois cas canoniques, un par chemin du programme :
  [-5, 2, 3]       -> verdict=True  pas=7 (attendu : verdict=True , pas=7)  [OK]
  [-3, -2, -1]     -> verdict=False pas=6 (attendu : verdict=False, pas=6)  [OK]
  [1, 2, 3]        -> verdict=False pas=7 (attendu : verdict=False, pas=7)  [OK]
Batterie seedee : 300/300 accords avec la definition de la source.
TEMOIN FERME : CONFIRME -- le compte de pas de l'interpreteur egale le compte a la main sur les trois chemins.


### Lecture du résultat — la machine et la main tombent d'accord

Sur les trois chemins (oui ; non par s < 0 ; non par s > 0), le compte de l'interpréteur égale le compte à la main, et sur la batterie seedée le verdict suit la définition `ThreeSum` de la source. Le témoin ne dit rien des cinq claims : il dit que **le modèle est bien celui qu'on a lu** — neuf instructions, un pas chacune, l'arrêt compté, le hors-programme qui rejette.

## 4. Le point de départ, mesuré — et pourquoi la borne ne se mesure pas

Le contrôle : le **3SUM quadratique** classique (tri + deux pointeurs, O(n²)) chronométré par doublement, sur des instances **sans solution** (trois nombres positifs ne somment jamais à 0) — sinon le chrono mesure « où se trouve le premier triple trouvé », pas le balayage complet ; c'est la leçon de méthode de cette section. Ce point de départ est celui que les bornes cassées devancent — **témoin du point de départ, jamais une mesure du gain annoncé**. Ce contrôle **ne réfute rien** : une pente ≈ 2 est exactement ce que la borne O(n^1.9992) elle-même prédit pour tout n atteignable, parce que (README du projet) « The constants of the time bounds are not optimized; they are large, and for most theorems no value is given » — l'écart d'exposant 2 − 1,9992 = 0,0008 vaut un facteur ~1,007 à n = 8 000, sous le bruit de n'importe quelle chronométrie. La cellule calcule ce facteur au lieu de le supposer.

In [6]:
def trois_somme_quadratique(xs):
    """3SUM quadratique classique : tri + deux pointeurs par ancre."""
    t = sorted(xs)
    n = len(t)
    for i in range(n - 2):
        cible = -t[i]
        g, d = i + 1, n - 1
        while g < d:
            s = t[g] + t[d]
            if s == cible:
                return True
            if s < cible:
                g += 1
            else:
                d -= 1
    return False

# Temoin de COUT COMPLET : instances SANS solution (trois positifs ne somment jamais a 0),
# sinon le chrono mesure « ou se trouve le premier triple », pas le balayage quadratique.
random.seed(SEED)
tailles = [1000, 2000, 4000, 8000]
REPS = 3   # mediane de 3 passages : les petites tailles sont sous le bruit d'une passe unique
trois_somme_quadratique([1, 2, 3])   # passe a blanc : caches et allocs hors mesure
mesures = []
for n in tailles:
    xs = [random.randint(1, n * n) for _ in range(n)]   # tous positifs : reponse False assuree
    temps = []
    for _ in range(REPS):
        dep = time.perf_counter()
        rep = trois_somme_quadratique(xs)
        temps.append(time.perf_counter() - dep)
    dt = sorted(temps)[REPS // 2]
    mesures.append((n, dt))
    print(f"n = {n:5d} : {dt * 1000:8.1f} ms (mediane de {REPS}, reponse {rep})")

# Pente par regression log-log globale, plus robuste que la moyenne des ratios par doublement
ln_n = [math.log(n) for n, _ in mesures]
ln_t = [math.log(t) for _, t in mesures]
mn, mt = sum(ln_n) / len(ln_n), sum(ln_t) / len(ln_t)
pente = (sum((a - mn) * (b - mt) for a, b in zip(ln_n, ln_t))
         / sum((a - mn) ** 2 for a in ln_n))
gain_8000 = 8000 ** (2 - 1.9992)
print()
print(f"Pente empirique du quadratique (regression log-log sur 4 tailles) : {pente:.3f} -- temoin du point de depart.")
print(f"Ce que n^(2-1.9992) vaut a n = 8000 : un facteur {gain_8000:.4f} -- sous le bruit de la mesure ci-dessus.")

n =  1000 :     60.2 ms (mediane de 3, reponse False)


n =  2000 :    246.0 ms (mediane de 3, reponse False)


n =  4000 :    971.9 ms (mediane de 3, reponse False)


n =  8000 :   4149.3 ms (mediane de 3, reponse False)

Pente empirique du quadratique (regression log-log sur 4 tailles) : 2.030 -- temoin du point de depart.
Ce que n^(2-1.9992) vaut a n = 8000 : un facteur 1.0072 -- sous le bruit de la mesure ci-dessus.


### Lecture du résultat — pente ≈ 2, un facteur 1,007 d'avance promise

La pente empirique du quadratique est ≈ 2 — c'est le **témoin** attendu. À n = 8 000, n^0,0008 ≈ 1,007 : même parfaite, l'implémentation de la borne cassée ne devancerait le quadratique que de 0,7 % — bien moins que l'écart entre deux exécutions de la même cellule. C'est la leçon du 07c sur le matmul en n^2,25 (crossover repoussé à ~n^10^16 si la constante vaut 100×) transposée : la victoire est **réelle en exposant** — elle existe comme théorème, et même comme programme word RAM —, **invisible en instance** tant que les constantes ne sont pas optimisées, et la source dit elle-même qu'elles ne le sont pas.

## 5. Ce que la source ne prouve pas

Quatre limites, chacune citée du [README](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/README.md) du projet (sections « What is not proved » et « The five claims ») :

1. **Les entrées réelles et la section 5.1 ne sont pas prouvées** — « the running times for real inputs (Theorem 35, Lemmas 36 and 37, Corollary 38, and the half of Theorem 2 on real inputs) and those of Section 5.1 (Theorems 33 and 34) » ne sont prouvées **pour aucune machine** : l'algorithme Las Vegas O(n^1.998)/O(n^2.998) du papier (p. 4) vit sur un real RAM que cette machine n'a pas.
2. **Les constantes ne sont pas optimisées** — « The constants of the time bounds are not optimized; they are large, and for most theorems no value is given. »
3. **Les opérations absentes de la machine** — « There is no division, no shift, no bitwise operation, and no constant other than 1. » Une borne prouvée sur cette machine est donc **plus forte** que la même borne sur un word RAM standard (docs/MACHINE.md : « an upper bound on time proved for it is stronger »).
4. **Le claim est une existence de programme** — « The claims say that programs exist. The proofs describe the programs, but not everything in them is computed » (le README renvoie à docs/MACHINE.md, « How explicit the programs are »).

Et deux prudences de plus, lues au même endroit : les hypothèses que le papier « réfute » (3SUM, APSP, Exact Triangle, k-Clique) **ne sont pas définies** dans la formalisation — seuls des temps de calcul sont énoncés ; et « This README records no run of the tool » — les auteurs décrivent la procédure de vérification Comparator mais n'en publient pas d'exécution. Ce carnet, de son côté, ne lance **aucun** `lake build`.

## 6. Plus faible ici, plus généreuse là

`docs/MACHINE.md` compare la machine au word RAM standard, point par point. **Plus faible en trois points** (donc des bornes plus fortes) : pas de division, décalage ni opération bit à bit ; **la seule constante est 1** — « a constant k has to be built, in about log k steps » ; le programme ne choisit pas sa taille de mot. **Aussi généreuse en deux** : une multiplication coûte un pas, et toute cellule dont le nom tient dans un mot est adressable. Quatre gratuités supplémentaires (cellules hors entrée à 0 sans coût, cellules nommées dans le texte hors de portée des mots, pente b choisie après κ, entrées de phases gratuites) — « none of them changes an exponent ».

La taille de mot se règle ainsi : le programme est fixé **avec** un nombre b, et doit tenir sa borne à toute taille W ≥ b·(1 + ⌊log₂ n⌋) — c'est le « O(log n)-bit words » du papier (Theorem 2), rendu à la lettre dans `SolvedInTime` (lignes 67–70).

## 7. Place dans la série

- **07c** (14 octobre) écrivait : « aucun de ces résultats (UGC, L=BPL, parité ∉ QAC⁰, exposants matmul/entiers, 3SUM sous-quadratique) n'a de formalisation référencée dans ce dépôt ». Cette phrase était vraie à sa date ; le projet `3sum-apsp` la rend périmée pour le 3SUM — un renvoi daté est ajouté dans 07c par la même PR que ce carnet.
- **03d**, le carnet frère : l'autre annonce d'octobre 2026 (la réfutation de l'ETH), lue sur ses trois surfaces Lean. Ici comme là-bas, la citation croisée se mesure : le **papier cite sa formalisation** (p. 63 : « The following are formalized there: Theorem 19… Theorem 22… Corollary 39 ») et la **formalisation cite le papier** — les deux sens, contrairement au cas ETH où la cross-citation n'allait que dans un sens.
- **Le gap Mathlib** (03b) : pas de `TIME(f)` dans Mathlib — et ce projet montre la réponse des auteurs : définir machine, sémantique et bornes **dans le noyau Lean seul**, sans Mathlib dans l'énoncé (`EndStatement.lean` n'importe rien). Le prix : un BigO rationnel (§1) et une machine minimaliste (§2) ; le gain : l'énoncé entier des cinq claims tient dans 139 lignes vérifiables.

Le papier ajoute, en son propre acknowledgments (p. 63) : « The authors used Claude to help with writing, figure creation, and checking mathematical details throughout this project. They have attempted to make the entire paper accessible and well-written, and take full responsibility » — et p. 1 : « Claude also verified this paper's main results using the Lean 4 proof assistant with the Mathlib library ».

## 8. Exercices

Trois exercices, sur le modèle de la série — stubs non bloquants (C.1) : le carnet s'exécute de bout en bout même non complété.

In [7]:
def construire_constante(k):
    """Exercice 1. La seule constante de la machine est 1 (instruction `one`) ;
    docs/MACHINE.md : « a constant k has to be built, in about log k steps ».
    Ecrire le programme (liste d'instructions comme P_3SUM) qui laisse k dans
    une cellule donnee, par doubles et additions.
    Retour attendu : la liste d'instructions, ex. [("one", 1), ("add", 1, 1, 1)] pour k = 2.
    """
    # TODO etudiant : construire k par une chaine de doubles puis decomposition binaire
    # Indice : 1 -> 2 -> 4 -> ... -> 2^j, puis k en binaire
    # Etape 1 : le cas k = 2^j
    # Etape 2 : generaliser
    print("Exercice 1 : construire_constante definie -- a completer (retour attendu : la liste d'instructions).")
    return None

construire_constante(10)

Exercice 1 : construire_constante definie -- a completer (retour attendu : la liste d'instructions).


**Exercice 2** — traduire le contrôle de la section 4 en **pas de machine** : le décideur quadratique vu par `exec`.

In [8]:
def bornes_pas_3sum_quadratique(n):
    """Exercice 2. Combien de PAS `exec` coute le decideur tri + deux pointeurs
    sur n nombres ? Chaque operation Python raisonnable devient une sequence
    d'instructions word RAM (comparer = sub + bltz, acceder = load, ...).
    Retour attendu : un ordre de grandeur du nombre de pas en fonction de n.
    """
    # TODO etudiant
    # Indice : boucle externe x boucle interne x cout d'une iteration traduite
    # Etape 1 : couter une iteration du two-pointer en instructions
    # Etape 2 : multiplier, puis verifier la coherence avec le temoin ferme (section 3)
    print("Exercice 2 : bornes_pas_3sum_quadratique definie -- a completer (retour attendu : un ordre de grandeur).")
    return None

bornes_pas_3sum_quadratique(1000)

Exercice 2 : bornes_pas_3sum_quadratique definie -- a completer (retour attendu : un ordre de grandeur).


**Exercice 3** — les deux algorithmes de Theorem 22 : mesurer l'écart que le claim laisse sur la table.

In [9]:
def ecart_deux_algorithmes():
    """Exercice 3. Theorem 22 imprime deux numeros par probleme (section 1) :
    1.99923 / 2.99949 par Theorem 5, 1.9992 / 2.99942 par Corollary 26 -- le
    claim prend les plus petits. Calculer, pour chaque probleme, l'ecart
    d'exposant entre les deux algorithmes, et le facteur correspondant
    a n = 10^6 puis n = 10^12.
    Retour attendu : un dict {probleme: [(ecart, facteur_1e6, facteur_1e12)], ...}.
    """
    # TODO etudiant
    # Indice : facteur = n ** ecart, comme la section 4 pour 2 - 1.9992
    print("Exercice 3 : ecart_deux_algorithmes definie -- a completer (retour attendu : le dict des ecarts).")
    return None

ecart_deux_algorithmes()

Exercice 3 : ecart_deux_algorithmes definie -- a completer (retour attendu : le dict des ecarts).


## 9. Conclusion

| Geste | Où | Ce qui a été fait | Niveau |
|---|---|---|---|
| Les cinq claims | §1 | Noms Lean, items du papier, exposants tels qu'imprimés ; les deux numéros de Theorem 22 ; le BigO rationnel (2499/1250) | Lu (fichiers) |
| La machine | §2–§3 | Port Python de `exec`/`loadWords` ; témoin fermé : compte de pas machine = compte à la main sur les trois chemins ; 300 instances seedées d'accord avec `ThreeSum` | **Mesuré** |
| Le point de départ | §4 | 3SUM quadratique chronométré (pente ≈ 2) ; le facteur n^0,0008 calculé (~1,007 à n = 8 000) — la borne ne se mesure pas aujourd'hui | **Mesuré** (témoin) |
| Ce que la source ne prouve pas | §5 | Quatre limites citées + deux prudences (hypothèses non définies, aucun run de Comparator publié) | Cité |
| La machine contre le standard | §6 | 3 faiblesses / 2 générosités / 4 gratuités (docs/MACHINE.md) | Cité |
| Place dans la série | §7 | Renvoi daté dans 07c ; 03d le frère ; gap Mathlib répondu hors Mathlib | Cité |

La conclusion en une phrase : **cinq claims d'existence de programme sont énoncés sur une machine de neuf instructions définie dans le fichier même qui les porte — et la machine, réimplémentée, tombe d'accord avec le compte à la main pendant que la borne promise reste, à n atteignable, un facteur 1,007 invisible : réelle en exposant, hors de portée en instance, exactement comme la source l'écrit.**

### Références (copie archivée au gisement)

- **Article** — Josh Alman & Virginia Vassilevska Williams, *Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs*, arXiv:2610.06783v1, 5 octobre 2026, 76 p. : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\2026 - Alman Vassilevska Williams - Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs (arXiv 2610.06783v1).pdf`
- **Formalisation** (lue au 28 octobre 2026) : [anthropics/formal-math · 3sum-apsp](https://github.com/anthropics/formal-math/tree/main/3sum-apsp) — `EndStatement.lean` (139 lignes), `README.md`, `docs/MACHINE.md`
- **Dans la série** : [03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) · [07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) · [03b — Hartmanis & Stearns et le gap Mathlib](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb)

*Issue de suivi : #23140. Carnet écrit le 28 octobre 2026 ; les faits datés (formalisation, statut des preuves) sont ceux de cette date.*